# Scout Advanced Notebook

This notebook covers more advanced querying patterns for Scout's HL7 radiology data and tips for working with large result sets.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Create the Trino engine once; query() below reuses it for every call.
_engine = create_engine(
    f"trino://{os.environ['TRINO_USER']}@{os.environ['TRINO_HOST']}:{os.environ['TRINO_PORT']}/"
    f"{os.environ['TRINO_CATALOG']}/{os.environ['TRINO_SCHEMA']}?http_scheme={os.environ['TRINO_SCHEME']}"
)


def query(sql_str: str, params: dict | None = None) -> pd.DataFrame:
    """Run a SQL query against Scout's Trino. Returns a DataFrame.

    Use `:name` placeholders in the SQL with values in `params`. For list
    values use Trino's `contains(array, element)`, SQLAlchemy doesn't expand
    list params into SQL IN clauses by default.

    Example:
        query("SELECT * FROM reports_latest WHERE contains(:facilities, sending_facility)",
              params={"facilities": ["BJH", "WUSM"]})
    """
    return pd.read_sql(text(sql_str), _engine, params=params or {})

In [ ]:
# Define the facilities you have IRB approval for. 
# Queries below pass the facilities list as a parameter to filter the data to just those facilities.
FACILITIES = ['BJH', 'WUSM', 'BJCMG', 'SLCH']

In [ ]:
# Longitudinal brain MRI cohort — patients with brain/head MRIs spanning 5+ years.
longitudinal_brain = query("""
    SELECT ANY_VALUE(resolved_epic_mrn) AS epic_mrn,
           ANY_VALUE(resolved_mpi)      AS mpi,
           COUNT(*) AS brain_mri_count,
           MIN(requested_dt) AS earliest,
           MAX(requested_dt) AS latest,
           date_diff('year', MIN(requested_dt), MAX(requested_dt)) AS years_span
    FROM reports_latest_epic_view
    WHERE contains(:facilities, sending_facility)
      AND modality = 'MR'
      AND regexp_like(service_name, '(?i)(brain|head)')
    GROUP BY scout_patient_id
    HAVING COUNT(*) >= 2
       AND date_diff('year', MIN(requested_dt), MAX(requested_dt)) >= 5
    ORDER BY years_span DESC, brain_mri_count DESC
    LIMIT 10
""", params={"facilities": FACILITIES})
longitudinal_brain

In [ ]:
# Lung nodule surveillance — patients with any 3 chest CTs within an 18-month rolling
# window AND at least one impression mentioning a nodule.
lung_surveillance = query("""
    WITH chest_cts AS (
        SELECT scout_patient_id, requested_dt, report_section_impression,
               resolved_epic_mrn, resolved_mpi
        FROM reports_latest_epic_view
        WHERE contains(:facilities, sending_facility)
          AND modality = 'CT'
          AND regexp_like(service_name, '(?i)(chest|thorax|lung)')
    ),
    surveillance_bursts AS (
        SELECT scout_patient_id,
               MIN(requested_dt) AS burst_first_ct,
               MIN_BY(third_ct_dt, requested_dt) AS burst_third_ct
        FROM (
            SELECT scout_patient_id, requested_dt,
                   LEAD(requested_dt, 2) OVER (PARTITION BY scout_patient_id ORDER BY requested_dt) AS third_ct_dt
            FROM chest_cts
        )
        WHERE date_diff('month', requested_dt, third_ct_dt) <= 18
        GROUP BY scout_patient_id
    ),
    nodule_patients AS (
        SELECT DISTINCT scout_patient_id
        FROM chest_cts
        WHERE regexp_like(report_section_impression, '(?i)nodule')
    )
    SELECT ANY_VALUE(c.resolved_epic_mrn) AS epic_mrn,
           ANY_VALUE(c.resolved_mpi)      AS mpi,
           COUNT(*) AS total_chest_cts,
           COUNT(*) FILTER (WHERE regexp_like(c.report_section_impression, '(?i)nodule')) AS nodule_mentions,
           ANY_VALUE(s.burst_first_ct)  AS burst_first_ct,
           ANY_VALUE(s.burst_third_ct)  AS burst_third_ct
    FROM chest_cts c
    JOIN surveillance_bursts s ON c.scout_patient_id = s.scout_patient_id
    JOIN nodule_patients n ON c.scout_patient_id = n.scout_patient_id
    GROUP BY c.scout_patient_id
    ORDER BY nodule_mentions DESC, total_chest_cts DESC
    LIMIT 10
""", params={"facilities": FACILITIES})
lung_surveillance

In [ ]:
# Ischemic stroke patients with all their prior imaging summarized.
# CTE finds the patients first stroke diagnosis, then joins back to all their prior imaging for summary stats.
stroke_history = query("""
    WITH stroke_patients AS (
        SELECT scout_patient_id,
               MIN(requested_dt) AS first_stroke_dt
        FROM reports_latest_epic_view
        WHERE contains(:facilities, sending_facility)
          AND any_match(diagnoses, d -> d.diagnosis_code LIKE 'I63%')
        GROUP BY scout_patient_id
        LIMIT 100
    )
    SELECT ANY_VALUE(r.resolved_epic_mrn) AS epic_mrn,
           ANY_VALUE(r.resolved_mpi)      AS mpi,
           COUNT(*) AS prior_reports,
           MIN(r.requested_dt) AS earliest_imaging,
           MAX(r.requested_dt) AS latest_imaging,
           array_sort(array_agg(DISTINCT r.modality)) AS modalities
    FROM reports_latest_epic_view r
    JOIN stroke_patients sp ON r.scout_patient_id = sp.scout_patient_id
    WHERE contains(:facilities, r.sending_facility)
      AND r.requested_dt < sp.first_stroke_dt
    GROUP BY r.scout_patient_id
    ORDER BY prior_reports DESC
    LIMIT 10
""", params={"facilities": FACILITIES})
stroke_history

## Streaming large result sets

If the full DataFrame from a query is too large to fit in memory, you can still process it in batches without materializing the whole thing.

In [ ]:
import trino
from collections import Counter


def needs_followup(impression: str | None) -> bool:
    """Mock classifier. A real version would call an ML model or annotation service."""
    if not impression:
        return False
    text = impression.lower()
    return any(phrase in text for phrase in (
        "recommend follow", "follow-up", "follow up", "repeat imaging",
    ))


# `with` blocks make sure the connection and cursor are closed even if processing
# raises mid-loop.
with trino.dbapi.connect(
    host=os.environ['TRINO_HOST'],
    port=int(os.environ['TRINO_PORT']),
    user=os.environ['TRINO_USER'],
    catalog=os.environ['TRINO_CATALOG'],
    schema=os.environ['TRINO_SCHEMA'],
    http_scheme=os.environ['TRINO_SCHEME'],
    encoding="json+zstd",
) as conn:
    with conn.cursor() as cur:
        # Bounded with LIMIT for the demonstration.
        cur.execute("""
            SELECT accession_number, modality, report_section_impression
            FROM reports_latest
            WHERE contains(?, sending_facility) AND year = 2024
            LIMIT 1000
        """, (FACILITIES,))

        # Each row is a tuple matching the SELECT column order.
        followups_by_modality = Counter()
        total = 0
        while True:
            rows = cur.fetchmany(100)
            if not rows:
                break
            for (accession, modality, impression) in rows:
                total += 1
                if needs_followup(impression):
                    followups_by_modality[modality] += 1

print(f"Streamed {total:,} reports. Follow-up mentions by modality:")
for mod, count in followups_by_modality.most_common(10):
    print(f"  {mod}: {count:,}")

## Using the Scout LLM

Scout hosts Ollama at `http://ollama.scout-analytics:11434` (cluster-internal DNS) and can be called from a notebook.

In [ ]:
# Pull a brain MRI report with an ischemic stroke diagnosis (I63%)
# and ask the LLM to summarize the full report text.
import requests

OLLAMA_URL = "http://ollama.scout-analytics:11434"
OLLAMA_MODEL = "gemma4-31b-long:latest"

sample = query("""
    SELECT accession_number, requested_dt, report_text
    FROM reports_latest
    WHERE contains(:facilities, sending_facility)
      AND modality = 'MR'
      AND regexp_like(service_name, '(?i)(brain|head)')
      AND any_match(diagnoses, d -> d.diagnosis_code LIKE 'I63%')
      AND report_text IS NOT NULL
      AND year = 2024
    LIMIT 1
""", params={"facilities": FACILITIES})

row = sample.iloc[0]
print(f"--- Report (accession {row['accession_number']}, {row['requested_dt']}) ---")
print(row["report_text"])

response = requests.post(
    f"{OLLAMA_URL}/api/generate",
    json={
        "model": OLLAMA_MODEL,
        "prompt": f"Summarize this radiology report in 2-3 sentences:\n\n{row['report_text']}",
        "stream": False,
        "options": {"temperature": 0},
    },
    timeout=120,
)
response.raise_for_status()
print("\n--- LLM summary ---")
print(response.json()["response"].strip())